# Video Enhancer — Colab worker (production)
**EN:** Processes the Video Enhancer queue on a free Colab T4 GPU. Run it by hand: *Runtime -> Run all*.
It claims queued jobs one at a time, enhances each video, uploads the result, and **exits when the queue is empty**.
There is no keep-alive, polling loop or background daemon (Colab policy). To process new jobs, run the notebook again.

**TH:** โน้ตบุ๊กนี้ประมวลผลคิวของ Video Enhancer บน Colab T4 GPU ฟรี ให้กด *Runtime -> Run all* ด้วยตนเอง
ระบบจะรับงานทีละงาน ปรับปรุงวิดีโอ อัปโหลดผลลัพธ์ แล้ว**จบการทำงานเมื่อคิวว่าง**
ไม่มี keep-alive, polling หรือ daemon เบื้องหลัง (ตามนโยบาย Colab) ถ้ามีงานใหม่ให้รันโน้ตบุ๊กอีกครั้ง

### Prerequisites — สิ่งที่ต้องเตรียม
1. **Runtime -> Change runtime type -> T4 GPU** (ตั้ง Runtime เป็น T4 GPU)
2. **Colab Secrets** (the key icon in the left sidebar — ไอคอนกุญแจด้านซ้าย), turn on **Notebook access** for both:
   - `ENHANCER_TOKEN` — worker token from the web app **Settings** page (operator only; generating a new one revokes the old one)
     — โทเคนจากหน้า Settings ของเว็บแอป (เฉพาะผู้ดูแล; สร้างใหม่แล้วโทเคนเก่าจะใช้ไม่ได้)
   - `ENHANCER_API` — base URL of the web app, e.g. `https://your-app.vercel.app`
     — URL หลักของเว็บแอป
3. Optional: Google Drive access to cache model weights between sessions (กด Allow เมื่อมีหน้าต่างขอสิทธิ์ Drive)

Secrets are never printed; only a masked token and the API host are shown. — ความลับจะไม่ถูกพิมพ์ออกมา

## 1. Settings — ตั้งค่า
**EN:** Edit the form fields, then run all cells. **TH:** แก้ค่าในฟอร์ม แล้วรันทุกเซลล์
- `REPO_REF`: branch / tag / commit of the worker code to install — เวอร์ชันโค้ดที่จะติดตั้ง
- `USE_DRIVE_CACHE`: keep model weights in Google Drive so later sessions skip the download — เก็บ weights ใน Drive
- `MAX_JOBS`: `0` = run until the queue is empty — `0` คือรันจนคิวว่าง
- `RUNNER`: `realesrgan` (real GPU runner) or `mock` (test only, no GPU) — `mock` ใช้ทดสอบเท่านั้น
- `CPU_FALLBACK`: if the session has no GPU, run on the CPU instead of stopping. It claims **only Anime 4× jobs** and is
  much slower (roughly 15–30×); other jobs stay queued for a GPU session —
  ถ้าไม่มี GPU ให้ใช้ CPU แทนการหยุด จะรับเฉพาะงาน **Anime 4×** และช้ากว่ามาก (ประมาณ 15–30 เท่า) งานอื่นยังรอคิวสำหรับ session ที่มี GPU

In [ ]:
REPO_REF = "main"  # @param {type:"string"}
USE_DRIVE_CACHE = True  # @param {type:"boolean"}
WEIGHTS_DIR_DRIVE = "/content/drive/MyDrive/enhancer-weights"  # @param {type:"string"}
MAX_JOBS = 0  # @param {type:"integer"}
RUNNER = "realesrgan"  # @param ["realesrgan", "mock"]
CPU_FALLBACK = True  # @param {type:"boolean"}

import re
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/moking55/video-enhancer"
WEIGHTS_DIR_LOCAL = "/content/weights"
WORK_DIR = "/content/enhancer-work"

if RUNNER not in ("realesrgan", "mock"):
    raise RuntimeError(f"RUNNER must be realesrgan or mock, got {RUNNER!r} / RUNNER ต้องเป็น realesrgan หรือ mock")
if not isinstance(MAX_JOBS, int) or MAX_JOBS < 0:
    raise RuntimeError("MAX_JOBS must be an integer >= 0 / MAX_JOBS ต้องเป็นจำนวนเต็ม 0 ขึ้นไป")
if not re.fullmatch(r"[A-Za-z0-9._/\-]{1,200}", REPO_REF):
    raise RuntimeError("REPO_REF has unexpected characters / REPO_REF มีตัวอักษรที่ไม่ถูกต้อง")
print(f"runner={RUNNER} ref={REPO_REF} max_jobs={MAX_JOBS or 'until queue empty'} drive_cache={USE_DRIVE_CACHE} "
      f"cpu_fallback={CPU_FALLBACK}")

## 2. GPU check — ตรวจ GPU
**EN:** With no GPU, the `realesrgan` runner either falls back to the CPU (`CPU_FALLBACK` on: Anime 4× jobs only, much slower)
or stops with a clear message. **TH:** ถ้าไม่มี GPU จะใช้ CPU แทน (เมื่อเปิด `CPU_FALLBACK`: เฉพาะงาน Anime 4× และช้ากว่ามาก) หรือหยุดพร้อมข้อความชัดเจน

In [ ]:
def _run(cmd, timeout=60):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    except (OSError, subprocess.SubprocessError) as e:
        return subprocess.CompletedProcess(cmd, 127, "", f"{type(e).__name__}: {e}")


smi = _run(["nvidia-smi"], timeout=20)
HAS_GPU = smi.returncode == 0
if HAS_GPU:
    print(smi.stdout)
else:
    print("nvidia-smi: no GPU found / ไม่พบ GPU")

DEVICE = "cuda" if HAS_GPU else "cpu"
if RUNNER == "realesrgan" and not HAS_GPU:
    if CPU_FALLBACK:
        print(
            "WARNING: no GPU - using the CPU instead. Only Anime 4x jobs will be claimed and it is much slower "
            "(roughly 15-30x); other jobs stay queued for a GPU session.\n"
            "ไม่มี GPU — ใช้ CPU แทน จะรับเฉพาะงาน Anime 4× และช้ากว่ามาก (ประมาณ 15–30 เท่า) "
            "งานอื่นยังคงรอคิวสำหรับ session ที่มี GPU"
        )
        DEVICE = "cpu"
    else:
        raise RuntimeError(
            "No GPU. Use Runtime -> Change runtime type -> T4 GPU, then Run all again "
            "(or turn on CPU_FALLBACK).\n"
            "ไม่พบ GPU: ไปที่ Runtime -> Change runtime type -> T4 GPU แล้วกด Run all ใหม่ (หรือเปิด CPU_FALLBACK)"
        )

## 3. Secrets — อ่านความลับจาก Colab Secrets
**EN:** Reads `ENHANCER_TOKEN` and `ENHANCER_API`, validates them, prints only a masked token and the API host.
**TH:** อ่านและตรวจความถูกต้อง แสดงเฉพาะโทเคนแบบปิดบังและชื่อโฮสต์ของ API

In [ ]:
from urllib.parse import urlparse

try:
    from google.colab import userdata
except ImportError as e:
    raise RuntimeError("This notebook must run in Google Colab / ต้องรันบน Google Colab") from e


def _secret(name):
    try:
        value = userdata.get(name)
    except Exception:  # SecretNotFoundError, NotebookAccessError, ...
        raise RuntimeError(
            f"Secret {name} is missing or notebook access is off (key icon, left sidebar).\n"
            f"ไม่พบ Secret {name} หรือยังไม่เปิด Notebook access (ไอคอนกุญแจด้านซ้าย)"
        ) from None
    value = (value or "").strip()
    if not value:
        raise RuntimeError(f"Secret {name} is empty / Secret {name} ว่างเปล่า")
    return value


ENHANCER_TOKEN = _secret("ENHANCER_TOKEN")
ENHANCER_API = _secret("ENHANCER_API").rstrip("/")

if not re.fullmatch(r"ve_[A-Za-z0-9_-]{43}", ENHANCER_TOKEN):
    raise RuntimeError(
        "ENHANCER_TOKEN does not look like a worker token (expected ve_ + 43 characters). "
        "Copy it again from the web app Settings page.\n"
        "ENHANCER_TOKEN รูปแบบไม่ถูกต้อง (ต้องขึ้นต้น ve_ ตามด้วย 43 ตัวอักษร) คัดลอกใหม่จากหน้า Settings"
    )
_api = urlparse(ENHANCER_API)
if _api.scheme != "https" or not _api.hostname or _api.username or _api.password or _api.query:
    raise RuntimeError(
        "ENHANCER_API must be a plain https URL like https://your-app.vercel.app (no credentials).\n"
        "ENHANCER_API ต้องเป็น URL แบบ https เช่น https://your-app.vercel.app (ห้ามมีชื่อผู้ใช้/รหัสผ่าน)"
    )

MASKED_TOKEN = "ve_..." + ENHANCER_TOKEN[-4:]


def redact(text):
    """Remove the token from any text before it is shown."""
    return text.replace(ENHANCER_TOKEN, MASKED_TOKEN)


print(f"token: {MASKED_TOKEN}")
print(f"api host: {_api.hostname}")

## 4. Install worker code — ติดตั้งโค้ด worker
**EN:** `pip install` from the public GitHub repo at `REPO_REF`. **TH:** ติดตั้งจาก GitHub repo สาธารณะตาม `REPO_REF`

In [ ]:
import importlib
import json
from importlib import metadata

spec = f"git+{REPO_URL}@{REPO_REF}#subdirectory=worker"
p = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", spec],
    capture_output=True, text=True,
)
if p.returncode != 0:
    print(redact((p.stdout + p.stderr)[-2000:]))
    raise RuntimeError(
        f"pip install failed (check REPO_REF={REPO_REF!r} and network).\n"
        f"ติดตั้งไม่สำเร็จ (ตรวจ REPO_REF={REPO_REF!r} และอินเทอร์เน็ต)"
    )
importlib.invalidate_caches()

_dist = metadata.distribution("enhancer-worker")
_commit = ""
try:
    _commit = json.loads(_dist.read_text("direct_url.json") or "{}").get("vcs_info", {}).get("commit_id", "")
except Exception:
    pass
print(f"enhancer-worker {_dist.version}" + (f" @ {_commit[:10]}" if _commit else ""))

## 5. Weights cache (Google Drive) — แคช weights
**EN:** Mounts Google Drive when `USE_DRIVE_CACHE` is on (a permission prompt appears); otherwise uses `/content/weights`.
**TH:** เมานต์ Google Drive เมื่อเปิด `USE_DRIVE_CACHE` (จะมีหน้าต่างขอสิทธิ์) ไม่เช่นนั้นใช้ `/content/weights`

In [ ]:
WEIGHTS_DIR = WEIGHTS_DIR_LOCAL
if USE_DRIVE_CACHE:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        WEIGHTS_DIR = WEIGHTS_DIR_DRIVE
    except Exception as e:
        print(f"Drive mount failed ({type(e).__name__}); using {WEIGHTS_DIR_LOCAL} instead.")
        print(f"เมานต์ Drive ไม่สำเร็จ ใช้ {WEIGHTS_DIR_LOCAL} แทน (ต้องดาวน์โหลด weights ใหม่ทุก session)")
print(f"weights dir: {WEIGHTS_DIR}")

## 6. Preflight — เตรียม Real-ESRGAN และ weights
**EN:** Installs Real-ESRGAN (pinned) and downloads + verifies the weights of every preset, so the first job does not stall.
**TH:** ติดตั้ง Real-ESRGAN และดาวน์โหลด weights ของทุก preset พร้อมตรวจ SHA-256 เพื่อไม่ให้งานแรกค้าง

In [ ]:
if RUNNER != "realesrgan":
    print("mock runner: nothing to prepare / ไม่ต้องเตรียมอะไร")
else:
    from enhancer_worker.setup_colab import ensure_realesrgan
    from enhancer_worker.runners.realesrgan import MODELS
    from enhancer_worker.weights import WEIGHTS, ensure_weights, sha256_file

    ensure_realesrgan()
    for name in dict.fromkeys(m.weights for m in MODELS.values()):
        path = ensure_weights(name, WEIGHTS_DIR, log=print)
        digest = sha256_file(path)
        pinned = WEIGHTS[name].sha256
        status = "OK" if pinned is None or digest == pinned else "MISMATCH"
        print(f"{path.name}: sha256 {digest[:12]}... {status}")
        if status != "OK":
            raise RuntimeError(f"Weights hash mismatch for {path.name}; delete it from {WEIGHTS_DIR} and retry.")
    print("preflight OK / พร้อมแล้ว")

## 7. Connectivity check — ตรวจการเชื่อมต่อ
**EN:** The worker API has no read-only endpoint (every call changes state, `claim` would take a job), so the token is **not** tested here.
This cell only checks that the web app is reachable (no token sent). A bad token shows up as exit code 1 in the run step.
**TH:** API ของ worker ไม่มี endpoint แบบอ่านอย่างเดียว (ทุกการเรียกเปลี่ยนสถานะ และ `claim` จะดึงงานไปทำ) จึงไม่ทดสอบโทเคนที่นี่
เซลล์นี้ตรวจแค่ว่าเว็บแอปเข้าถึงได้ (ไม่ส่งโทเคน) ถ้าโทเคนผิดจะเห็นเป็น exit code 1 ในขั้นรัน

In [ ]:
import requests

try:
    r = requests.get(ENHANCER_API, timeout=15, allow_redirects=False)
    print(f"web app reachable (HTTP {r.status_code}) / เข้าถึงเว็บแอปได้")
except requests.RequestException as e:
    raise RuntimeError(
        f"Cannot reach the web app ({type(e).__name__}). Check the ENHANCER_API secret.\n"
        f"เข้าถึงเว็บแอปไม่ได้ ({type(e).__name__}) ตรวจ Secret ENHANCER_API"
    ) from None

## 8. Run the queue — ประมวลผลคิว
**EN:** Runs `python -m enhancer_worker run` as a subprocess. The token goes through the environment, not the command line.
Output streams below. It ends when the queue is empty (or `MAX_JOBS` is reached).
**TH:** รัน worker เป็น subprocess ส่งโทเคนผ่าน environment ไม่ใช่ argument แสดงผลทีละบรรทัด จบเมื่อคิวว่าง (หรือครบ `MAX_JOBS`)

In [ ]:
import os

os.makedirs(WORK_DIR, exist_ok=True)
cmd = [
    sys.executable, "-u", "-m", "enhancer_worker", "run",
    "--api", ENHANCER_API,
    "--runner", RUNNER,
    "--device", DEVICE,
    "--weights-dir", WEIGHTS_DIR,
    "--work-dir", WORK_DIR,
]
if MAX_JOBS > 0:
    cmd += ["--max-jobs", str(MAX_JOBS)]

env = dict(os.environ, ENHANCER_TOKEN=ENHANCER_TOKEN, PYTHONUNBUFFERED="1")
JOB_COUNTS = {"succeeded": 0, "failed": 0, "cancelled": 0}
_job_re = re.compile(r"^job \S+: (succeeded|failed|cancelled)\b")

proc = subprocess.Popen(
    cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, errors="replace",
)
try:
    for raw in proc.stdout:
        line = redact(raw.rstrip("\n"))
        print(line, flush=True)
        m = _job_re.match(line)
        if m:
            JOB_COUNTS[m.group(1)] += 1
    EXIT_CODE = proc.wait()
except KeyboardInterrupt:
    proc.terminate()
    try:
        proc.wait(timeout=30)
    except subprocess.TimeoutExpired:
        proc.kill()
    EXIT_CODE = -1
    print("Interrupted by the user; a running job will be requeued by its lease timeout.")
    print("ถูกหยุดโดยผู้ใช้ งานที่กำลังทำจะถูกคืนเข้าคิวเมื่อ lease หมดเวลา")

print()
if EXIT_CODE == 0:
    print("DONE: queue is empty (or MAX_JOBS reached).")
    print("เสร็จสิ้น: คิวว่างแล้ว (หรือครบ MAX_JOBS)")
elif EXIT_CODE == 1:
    print("AUTH/API ERROR: the token was rejected or the API failed. Generate a new token on the web app Settings page,")
    print("update the ENHANCER_TOKEN secret, then run again.")
    print("โทเคนไม่ถูกยอมรับหรือ API ผิดพลาด: สร้างโทเคนใหม่ที่หน้า Settings อัปเดต Secret ENHANCER_TOKEN แล้วรันใหม่")
elif EXIT_CODE == 2:
    print("SETUP ERROR (exit 2): see the log above. / ตั้งค่าหรือติดตั้งไม่สำเร็จ (exit 2) ดู log ด้านบน")
else:
    print(f"STOPPED with exit code {EXIT_CODE}: see the log above. / หยุดด้วย exit code {EXIT_CODE} ดู log ด้านบน")

## 9. Cleanup and summary — ล้างไฟล์ชั่วคราวและสรุป
**EN:** Deletes the scratch directory and prints the job counts.
**TH:** ลบโฟลเดอร์ชั่วคราวและสรุปจำนวนงาน

In [ ]:
shutil.rmtree(WORK_DIR, ignore_errors=True)
print(f"scratch removed: {not os.path.exists(WORK_DIR)} / ลบไฟล์ชั่วคราวแล้ว")

total = sum(JOB_COUNTS.values())
print(
    f"jobs processed: {total} "
    f"(succeeded {JOB_COUNTS['succeeded']}, failed {JOB_COUNTS['failed']}, cancelled {JOB_COUNTS['cancelled']}) / "
    f"จำนวนงานที่ประมวลผล: {total}"
)
print()
print("To process new jobs, re-run this notebook (Runtime -> Run all).")
print("ถ้ามีงานใหม่ ให้รันโน้ตบุ๊กนี้อีกครั้ง (Runtime -> Run all)")
print()
print("To free the GPU: Runtime -> Disconnect and delete runtime.")
print("เพื่อคืน GPU: Runtime -> Disconnect and delete runtime")